# End-to-End Supervised Learning Workflow: HDB Resale Price Prediction with CatBoost

**Goal**: Build a supervised regression workflow to predict `resale_price` from the training dataset using `CatBoostRegressor`.

**Scenario**

The dataset contains flat attributes, transaction timing, address/location signals, nearby amenities, schools, and transport features. The target variable is `resale_price`, so this is a supervised regression problem evaluated with RMSE.

**Workflow**

1. Load the sample submission, test data, and training data.
2. Define sklearn-compatible feature transformers for transaction timing, lease/age/storey features, flat-type ordinal encoding, distance/accessibility features, categorical combinations, spatial indexing, and HDBSCAN geo-cluster features.
3. Convert each latitude/longitude pair into broad-to-fine S2 cell and geohash area labels for the flat, nearest MRT, nearest bus stop, nearest primary school, and nearest secondary school.
4. Split the labeled training data into features (`X`) and target (`Y`).
5. Define preprocessing groups: dropped identifiers/raw coordinate columns, regular categorical columns, spatial-index categorical columns, binary columns, continuous numeric columns, and remaining numeric columns.
6. Build a reusable pipeline that creates engineered features, drops unused/raw coordinate columns, imputes missing values, scales numeric features, and passes categorical plus spatial-index features to CatBoost for native categorical handling.
7. Create a held-out train-test split, then evaluate a baseline CatBoost model with 10-fold cross-validation on the training split.
8. Fit the baseline CatBoost model and evaluate it on the held-out test split.
9. Tune CatBoost hyperparameters with CatBoost's built-in grid search, refit the tuned pipeline, and compare against the baseline.
10. Fit the final tuned pipeline on all labeled training data, predict the Kaggle test data, save the model, and write the submission CSV.


## 1. Imports and Data Loading

First, import the libraries used throughout the workflow and load the training dataset. The `id` column is used as the dataframe index.


In [68]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor, Pool
from s2sphere import CellId, LatLng
import geohash2
import hdbscan
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

pd.set_option('display.max_columns', 120)


In [69]:
# load submission format
df_sub = pd.read_csv('data/sample_sub_reg.csv')

df_sub.shape
df_sub.head()


,Id,Predicted
0,114982,500000
1,95653,500000
2,40303,500000
3,109506,500000
4,100149,500000


In [70]:
# load test dataset
df_test = pd.read_csv('data/test.csv')

df_test.shape
df_test.head()

C:\Users\zacang\AppData\Local\Temp\ipykernel_14288\3630737951.py:2: DtypeWarning: Columns (0: postal) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('data/test.csv')


,id,Tranc_YearMonth,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,Tranc_Year,Tranc_Month,mid_storey,lower,upper,mid,full_flat_type,address,floor_area_sqft,hdb_age,max_floor_lvl,year_completed,residential,commercial,market_hawker,multistorey_carpark,precinct_pavilion,total_dwelling_units,1room_sold,2room_sold,3room_sold,4room_sold,5room_sold,exec_sold,multigen_sold,studio_apartment_sold,1room_rental,2room_rental,3room_rental,other_room_rental,postal,Latitude,Longitude,planning_area,Mall_Nearest_Distance,Mall_Within_500m,Mall_Within_1km,Mall_Within_2km,Hawker_Nearest_Distance,Hawker_Within_500m,Hawker_Within_1km,Hawker_Within_2km,hawker_food_stalls,hawker_market_stalls,mrt_nearest_distance,mrt_name,bus_interchange,mrt_interchange,mrt_latitude,mrt_longitude,bus_stop_nearest_distance,bus_stop_name,bus_stop_latitude,bus_stop_longitude,pri_sch_nearest_distance,pri_sch_name,vacancy,pri_sch_affiliation,pri_sch_latitude,pri_sch_longitude,sec_sch_nearest_dist,sec_sch_name,cutoff_point,affiliation,sec_sch_latitude,sec_sch_longitude
0,114982,2012-11,YISHUN,4 ROOM,173,YISHUN AVE 7,07 TO 09,84.0,Simplified,1987,2012,11,8,7,9,8,4 ROOM Simplified,"173, YISHUN AVE 7",904.176,34,12,1986,Y,Y,N,N,N,132,0,0,0,92,40,0,0,0,0,0,0,0,760173,1.437066,103.831121,Yishun,877.431572,NaN,2.0,4.0,687.576779,NaN,1.0,1.0,56,123,686.660434,Canberra,0,0,1.443077,103.829703,75.683952,Blk 174,1.437558,103.831591,426.467910,Ahmad Ibrahim Primary School,92,0,1.433681,103.832924,156.322353,Ahmad Ibrahim Secondary School,218,0,1.436235,103.829987
1,95653,2019-08,JURONG WEST,5 ROOM,986C,JURONG WEST ST 93,04 TO 06,112.0,Premium Apartment,2008,2019,8,5,4,6,5,5 ROOM Premium Apartment,"986C, JURONG WEST ST 93",1205.568,13,14,2002,Y,N,N,N,N,53,0,0,0,28,25,0,0,0,0,0,0,0,643986,1.336957,103.695668,Jurong West,534.037705,NaN,1.0,3.0,2122.346226,NaN,NaN,NaN,72,94,169.478175,Pioneer,0,0,1.337343,103.697143,88.993058,Blk 653B,1.336491,103.696319,439.756851,Jurong West Primary School,45,0,1.339244,103.698896,739.371688,Jurong West Secondary School,199,0,1.335256,103.702098
2,40303,2013-10,ANG MO KIO,3 ROOM,534,ANG MO KIO AVE 10,07 TO 09,68.0,New Generation,1980,2013,10,8,7,9,8,3 ROOM New Generation,"534, ANG MO KIO AVE 10",731.952,41,12,1979,Y,N,N,N,N,218,0,0,191,22,1,1,0,0,0,0,3,0,560534,1.374058,103.854168,Ang Mo Kio,817.050453,NaN,2.0,3.0,152.287621,1.0,3.0,11.0,50,100,694.220448,Ang Mo Kio,1,0,1.369465,103.849939,86.303575,Blk 532,1.374255,103.854919,355.882207,Jing Shan Primary School,36,0,1.371893,103.851811,305.071191,Anderson Secondary School,245,0,1.374242,103.851430
3,109506,2017-10,WOODLANDS,4 ROOM,29,MARSILING DR,01 TO 03,97.0,New Generation,1979,2017,10,2,1,3,2,4 ROOM New Generation,"29, MARSILING DR",1044.108,42,14,1976,Y,N,N,N,N,104,0,0,0,104,0,0,0,0,0,0,0,0,731029,1.442748,103.772922,Woodlands,1272.737194,NaN,NaN,3.0,501.892158,NaN,1.0,2.0,52,112,1117.203587,Marsiling,0,0,1.432757,103.773982,108.459039,Blk 32,1.443650,103.773295,929.744711,Marsiling Primary School,54,0,1.434423,103.773698,433.454591,Woodlands Secondary School,188,0,1.439183,103.774499
4,100149,2016-08,BUKIT BATOK,4 ROOM,170,BT BATOK WEST AVE 8,16 TO 18,103.0,Model A,1985,2016,8,17,16,18,17,4 ROOM Model A,"170, BT BATOK WEST AVE 8",1108.692,36,25,1985,Y,N,N,N,N,144,0,0,0,48,96,0,0,0,0,0,0,0,650170,1.346556,103.740101,Bukit Batok,1070.963675,NaN,NaN,5.0,437.593564,1.0,2.0,2.0,60,87,987.976010,Chinese Garden,0,0,1.342441,103.732225,113.645431,Blk 169,1.346899,103.741064,309.926934,Princess Elizabeth Primary School,40,0,1.349195,103.741000,217.295361,Bukit Batok Secondary School,223,0,1.348351,103.740873


In [71]:
# Load the training dataset.
df = pd.read_csv('data/train.csv')

df.shape

C:\Users\zacang\AppData\Local\Temp\ipykernel_14288\3207937961.py:2: DtypeWarning: Columns (0: postal) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/train.csv')


(150634, 77)

## 2. Feature Engineering Inside the Pipeline

Define feature engineering as a sklearn transformer so the model can receive raw data at both training time and prediction time. The transformer learns the first transaction month during `fit`, then creates transaction timing, lease, age, flat-type ordinal, and storey-ratio features during every `transform`.


In [72]:
class TransactionDateFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, transaction_col='Tranc_YearMonth'):
        self.transaction_col = transaction_col
        self.flat_type_ordinal_map = {
            '1 ROOM': 1,
            '2 ROOM': 2,
            '3 ROOM': 3,
            '4 ROOM': 4,
            '5 ROOM': 5,
            'EXECUTIVE': 6,
            'MULTI-GENERATION': 7,
            'MULTI GENERATION': 7,
        }
        self.engineered_features = [
            'days_from_first_transaction',
            'month_sin',
            'month_cos',
            'transaction_quarter',
            'remaining_lease',
            'property_age_at_sale',
            'is_new_flat',
            'is_old_flat',
            'flat_type_ordinal',
            'storey_mid_ratio',
            'log_mrt_dist',
            'log_mall_dist',
            'log_hawker_dist',
            'log_bus_dist',
            'log_pri_sch_dist',
            'log_sec_sch_dist',
            'accessibility_score',
            'mrt_walkable',
            'bus_walkable',
            'mall_walkable',
            'hawker_walkable',
            'pri_school_walkable',
            'sec_school_walkable',
            'integrated_transport_access',
            'town_full_flat_type',
            'block_street_name',
            'street_name_pri_sch_name',
            'street_name_sec_sch_name',
        ]

    def fit(self, X, y=None):
        transaction_dates = pd.to_datetime(X[self.transaction_col], errors='coerce')
        self.first_transaction_date_ = transaction_dates.min()
        return self

    def transform(self, X):
        X = X.copy()
        transaction_dates = pd.to_datetime(X[self.transaction_col], errors='coerce')
        trans_year = pd.to_numeric(X['Tranc_Year'], errors='coerce')
        trans_month = pd.to_numeric(X['Tranc_Month'], errors='coerce')
        lease_commence_date = pd.to_numeric(X['lease_commence_date'], errors='coerce')
        year_completed = pd.to_numeric(X['year_completed'], errors='coerce')
        mid_storey = pd.to_numeric(X['mid_storey'], errors='coerce')
        max_floor_lvl = pd.to_numeric(X['max_floor_lvl'], errors='coerce')
        mrt_distance = pd.to_numeric(X['mrt_nearest_distance'], errors='coerce')
        mall_distance = pd.to_numeric(X['Mall_Nearest_Distance'], errors='coerce')
        hawker_distance = pd.to_numeric(X['Hawker_Nearest_Distance'], errors='coerce')
        bus_distance = pd.to_numeric(X['bus_stop_nearest_distance'], errors='coerce')
        pri_sch_distance = pd.to_numeric(X['pri_sch_nearest_distance'], errors='coerce')
        sec_sch_distance = pd.to_numeric(X['sec_sch_nearest_dist'], errors='coerce')

        X['days_from_first_transaction'] = (
            transaction_dates - self.first_transaction_date_
        ).dt.days

        X['month_sin'] = np.sin(2 * np.pi * trans_month / 12)
        X['month_cos'] = np.cos(2 * np.pi * trans_month / 12)
        X['transaction_quarter'] = np.ceil(trans_month / 3)
        X['remaining_lease'] = 99 - (trans_year - lease_commence_date)
        X['property_age_at_sale'] = trans_year - year_completed
        X['is_new_flat'] = (X['property_age_at_sale'] <= 6).astype(int)
        X['is_old_flat'] = (X['remaining_lease'] <= 60).astype(int)
        X['flat_type_ordinal'] = (
            X['flat_type']
            .astype('string')
            .str.upper()
            .map(self.flat_type_ordinal_map)
        )
        X['storey_mid_ratio'] = mid_storey / max_floor_lvl.replace(0, np.nan)

        X['log_mrt_dist'] = np.log1p(mrt_distance)
        X['log_mall_dist'] = np.log1p(mall_distance)
        X['log_hawker_dist'] = np.log1p(hawker_distance)
        X['log_bus_dist'] = np.log1p(bus_distance)
        X['log_pri_sch_dist'] = np.log1p(pri_sch_distance)
        X['log_sec_sch_dist'] = np.log1p(sec_sch_distance)

        X['accessibility_score'] = (
            1 / (1 + mrt_distance)
            + 1 / (1 + mall_distance)
            + 1 / (1 + hawker_distance)
            + 1 / (1 + bus_distance)
        ) / 4

        X['mrt_walkable'] = (mrt_distance <= 500).astype(int)
        X['bus_walkable'] = (bus_distance <= 500).astype(int)
        X['mall_walkable'] = (mall_distance <= 500).astype(int)
        X['hawker_walkable'] = (hawker_distance <= 500).astype(int)
        X['pri_school_walkable'] = (pri_sch_distance <= 500).astype(int)
        X['sec_school_walkable'] = (sec_sch_distance <= 500).astype(int)
        X['integrated_transport_access'] = (
            X['bus_interchange'].astype(bool) & X['mrt_interchange'].astype(bool)
        ).astype(int)

        X['town_full_flat_type'] = X['town'].astype('string') + '_' + X['full_flat_type'].astype('string')
        X['block_street_name'] = X['block'].astype('string') + '_' + X['street_name'].astype('string')
        X['street_name_pri_sch_name'] = X['street_name'].astype('string') + '_' + X['pri_sch_name'].astype('string')
        X['street_name_sec_sch_name'] = X['street_name'].astype('string') + '_' + X['sec_sch_name'].astype('string')

        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return None
        return np.array(list(input_features) + self.engineered_features, dtype=object)



class SpatialIndexFeatures(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        coordinate_pairs=None,
        # Multi-resolution spatial layers: lower values capture broader
        # neighbourhoods, while higher values capture finer local variation.
        s2_levels=(12, 14, 16, 18),
        geohash_precisions=(5, 6, 7, 8),
    ):
        self.coordinate_pairs = coordinate_pairs or {
            'flat': ('Latitude', 'Longitude'),
            'mrt': ('mrt_latitude', 'mrt_longitude'),
            'bus_stop': ('bus_stop_latitude', 'bus_stop_longitude'),
            'pri_sch': ('pri_sch_latitude', 'pri_sch_longitude'),
            'sec_sch': ('sec_sch_latitude', 'sec_sch_longitude'),
        }
        self.s2_levels = tuple(s2_levels)
        self.geohash_precisions = tuple(geohash_precisions)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        for location_name, (latitude_col, longitude_col) in self.coordinate_pairs.items():
            latitudes = pd.to_numeric(X[latitude_col], errors='coerce')
            longitudes = pd.to_numeric(X[longitude_col], errors='coerce')
            valid_coordinates = latitudes.notna() & longitudes.notna()

            for level in self.s2_levels:
                feature_name = f'{location_name}_s2_cell_l{level}'
                X[feature_name] = 'missing'
                X.loc[valid_coordinates, feature_name] = [
                    str(CellId.from_lat_lng(LatLng.from_degrees(lat, lon)).parent(level).id())
                    for lat, lon in zip(
                        latitudes.loc[valid_coordinates],
                        longitudes.loc[valid_coordinates],
                    )
                ]

            for precision in self.geohash_precisions:
                feature_name = f'{location_name}_geohash_p{precision}'
                X[feature_name] = 'missing'
                X.loc[valid_coordinates, feature_name] = [
                    geohash2.encode(lat, lon, precision=precision)
                    for lat, lon in zip(
                        latitudes.loc[valid_coordinates],
                        longitudes.loc[valid_coordinates],
                    )
                ]

        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return None
        spatial_features = []
        for location_name in self.coordinate_pairs:
            spatial_features.extend([
                f'{location_name}_s2_cell_l{level}' for level in self.s2_levels
            ])
            spatial_features.extend([
                f'{location_name}_geohash_p{precision}' for precision in self.geohash_precisions
            ])
        return np.array(list(input_features) + spatial_features, dtype=object)


class GeoClusterFeatures(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        latitude_col='Latitude',
        longitude_col='Longitude',
        min_cluster_size=80,
        min_samples=20,
    ):
        self.latitude_col = latitude_col
        self.longitude_col = longitude_col
        self.min_cluster_size = min_cluster_size
        self.min_samples = min_samples
        self.engineered_features = ['geo_cluster', 'geo_cluster_prob', 'geo_noise_flag']

    def fit(self, X, y=None):
        coordinates = self._coordinate_frame(X)
        self.fallback_coordinates_ = coordinates.median()
        coordinates = coordinates.fillna(self.fallback_coordinates_)

        self.clusterer_ = hdbscan.HDBSCAN(
            min_cluster_size=self.min_cluster_size,
            min_samples=self.min_samples,
            prediction_data=True,
        )
        self.clusterer_.fit(coordinates)
        return self

    def transform(self, X):
        X = X.copy()
        coordinates = self._coordinate_frame(X).fillna(self.fallback_coordinates_)

        labels, probabilities = hdbscan.approximate_predict(self.clusterer_, coordinates)
        X['geo_cluster'] = pd.Series(labels, index=X.index).astype('string')
        X['geo_cluster_prob'] = probabilities
        X['geo_noise_flag'] = (labels == -1).astype(int)
        return X

    def _coordinate_frame(self, X):
        return pd.DataFrame({
            self.latitude_col: pd.to_numeric(X[self.latitude_col], errors='coerce'),
            self.longitude_col: pd.to_numeric(X[self.longitude_col], errors='coerce'),
        }, index=X.index)

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return None
        return np.array(list(input_features) + self.engineered_features, dtype=object)


class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns or []

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns, errors='ignore')

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return None
        return np.array([col for col in input_features if col not in self.columns], dtype=object)


## 3. Feature Selection and Target Split

Rows with missing values are removed instead of imputed. Then `resale_price` is separated as the target variable.


In [73]:
df_ml = df.copy()

Y = df_ml['resale_price']
X = df_ml.drop(columns=['resale_price'])

print(f'Original rows: {df.shape[0]}')
print(f'Rows kept for modeling: {df_ml.shape[0]}')
print(f'Features: {X.shape[1]}')
print(f'Target: {Y.name}')

X.sample(2)

Original rows: 150634
Rows kept for modeling: 150634
Features: 76
Target: resale_price


,id,Tranc_YearMonth,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,Tranc_Year,Tranc_Month,mid_storey,lower,upper,mid,full_flat_type,address,floor_area_sqft,hdb_age,max_floor_lvl,year_completed,residential,commercial,market_hawker,multistorey_carpark,precinct_pavilion,total_dwelling_units,1room_sold,2room_sold,3room_sold,4room_sold,5room_sold,exec_sold,multigen_sold,studio_apartment_sold,1room_rental,2room_rental,3room_rental,other_room_rental,postal,Latitude,Longitude,planning_area,Mall_Nearest_Distance,Mall_Within_500m,Mall_Within_1km,Mall_Within_2km,Hawker_Nearest_Distance,Hawker_Within_500m,Hawker_Within_1km,Hawker_Within_2km,hawker_food_stalls,hawker_market_stalls,mrt_nearest_distance,mrt_name,bus_interchange,mrt_interchange,mrt_latitude,mrt_longitude,bus_stop_nearest_distance,bus_stop_name,bus_stop_latitude,bus_stop_longitude,pri_sch_nearest_distance,pri_sch_name,vacancy,pri_sch_affiliation,pri_sch_latitude,pri_sch_longitude,sec_sch_nearest_dist,sec_sch_name,cutoff_point,affiliation,sec_sch_latitude,sec_sch_longitude
22522,72337,2018-04,JURONG WEST,4 ROOM,338A,TAH CHING RD,07 TO 09,94.0,Model A,2010,2018,4,8,7,9,8,4 ROOM Model A,"338A, TAH CHING RD",1011.816,11,20,2008,Y,N,N,N,N,117,0,0,0,117,0,0,0,0,0,0,0,0,611338,1.337815,103.722185,Jurong West,380.287175,1.0,1.0,3.0,352.215130,1.0,1.0,5.0,123,57,707.201432,Lakeside,0,0,1.344075,103.721063,128.981447,Opp Blk 364,1.337752,103.721026,463.798017,Lakeside Primary School,63,0,1.338376,103.718051,485.130272,Yuan Ching Secondary School,208,0,1.341892,103.720631
15779,164902,2014-09,WOODLANDS,5 ROOM,687A,WOODLANDS DR 75,01 TO 03,115.0,Premium Apartment,2003,2014,9,2,1,3,2,5 ROOM Premium Apartment,"687A, WOODLANDS DR 75",1237.860,18,14,2001,Y,N,N,N,N,161,0,0,0,0,161,0,0,0,0,0,0,0,731687,1.441004,103.807223,Woodlands,587.651561,NaN,1.0,5.0,728.653288,NaN,1.0,1.0,43,0,697.420035,Admiralty,0,0,1.440343,103.800984,202.647001,Blk 685A,1.441463,103.805459,304.996143,Greenwood Primary School,61,0,1.440110,103.804629,763.477160,Admiralty Secondary School,197,0,1.445891,103.802398


## 4. Building the Preprocessing Groups

The preprocessing groups are built in discrete steps:

1. Collect categorical, spatial-index, and binary columns.
2. From the remaining numeric columns, identify continuous columns.
3. The leftover numeric columns are treated as non-continuous numeric features.

Raw latitude/longitude are continuous coordinates, so they stay in the standard-scaled numeric group. S2 cells and geohashes are not continuous measurements; they are hierarchical area labels. Those spatial labels are therefore encoded as categorical features.


In [74]:
# Step 1: collect categorical, spatial-index, and binary columns.
raw_coordinate_cols = [
    'Latitude', 'Longitude',
    'mrt_latitude', 'mrt_longitude',
    'bus_stop_latitude', 'bus_stop_longitude',
    'pri_sch_latitude', 'pri_sch_longitude',
    'sec_sch_latitude', 'sec_sch_longitude',
]
drop_cols = [
    'id', 'postal', 'street_name', 'block', 'Tranc_YearMonth',
    'floor_area_sqft', 'address', 'storey_range','residential'
] + raw_coordinate_cols
model_input_cols = [col for col in X.columns if col not in drop_cols]

# S2/geohash values are spatial area IDs, not ordered numeric measurements.
# Multiple levels/precisions create broad-to-fine location layers.
# Keep them as categorical tokens so CatBoost can encode each area without
# implying that a larger cell ID or geohash string is numerically larger.
spatial_index_cols = SpatialIndexFeatures().get_feature_names_out([]).tolist()
engineered_categorical_cols = [
    'town_full_flat_type',
    'block_street_name',
    'street_name_pri_sch_name',
    'street_name_sec_sch_name',
    'geo_cluster',
]
categorical_cols = (
    X[model_input_cols].select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    + spatial_index_cols
    + engineered_categorical_cols
)

numeric_candidate_cols = X[model_input_cols].select_dtypes(include='number').columns.tolist()
binary_cols = [
    col for col in numeric_candidate_cols
    if X[col].nunique(dropna=True) == 2
]

print(f'Dropped columns ({len(drop_cols)}):')
print(drop_cols)
print()
print(f'Categorical columns ({len(categorical_cols)}):')
print(categorical_cols)
print()
print(f'Binary columns ({len(binary_cols)}):')
print(binary_cols)


Dropped columns (19):
['id', 'postal', 'street_name', 'block', 'Tranc_YearMonth', 'floor_area_sqft', 'address', 'storey_range', 'residential', 'Latitude', 'Longitude', 'mrt_latitude', 'mrt_longitude', 'bus_stop_latitude', 'bus_stop_longitude', 'pri_sch_latitude', 'pri_sch_longitude', 'sec_sch_latitude', 'sec_sch_longitude']

Categorical columns (58):
['town', 'flat_type', 'flat_model', 'full_flat_type', 'commercial', 'market_hawker', 'multistorey_carpark', 'precinct_pavilion', 'planning_area', 'mrt_name', 'bus_stop_name', 'pri_sch_name', 'sec_sch_name', 'flat_s2_cell_l12', 'flat_s2_cell_l14', 'flat_s2_cell_l16', 'flat_s2_cell_l18', 'flat_geohash_p5', 'flat_geohash_p6', 'flat_geohash_p7', 'flat_geohash_p8', 'mrt_s2_cell_l12', 'mrt_s2_cell_l14', 'mrt_s2_cell_l16', 'mrt_s2_cell_l18', 'mrt_geohash_p5', 'mrt_geohash_p6', 'mrt_geohash_p7', 'mrt_geohash_p8', 'bus_stop_s2_cell_l12', 'bus_stop_s2_cell_l14', 'bus_stop_s2_cell_l16', 'bus_stop_s2_cell_l18', 'bus_stop_geohash_p5', 'bus_stop_geohash

In [75]:
# Step 2: identify continuous numeric columns from the remaining numeric columns.
engineered_numeric_cols = (
    TransactionDateFeatures().engineered_features
    + ['geo_cluster_prob', 'geo_noise_flag']
)
numeric_cols = [col for col in numeric_candidate_cols if col not in binary_cols] + engineered_numeric_cols

continuous_cols = [
    col for col in numeric_cols
    if 'distance' in col.lower() or 'dist' in col.lower() or col in [
        'floor_area_sqm',
        'floor_area_sqft',
        'days_from_first_transaction',
        'month_sin',
        'month_cos',
        'remaining_lease',
        'property_age_at_sale',
        'storey_mid_ratio',
        'log_mrt_dist',
        'log_mall_dist',
        'log_hawker_dist',
        'log_bus_dist',
        'log_pri_sch_dist',
        'log_sec_sch_dist',
        'accessibility_score',
        'geo_cluster_prob',
    ]
]

print(f'Continuous numeric columns ({len(continuous_cols)}):')
print(continuous_cols)


Continuous numeric columns (21):
['floor_area_sqm', 'Mall_Nearest_Distance', 'Hawker_Nearest_Distance', 'mrt_nearest_distance', 'bus_stop_nearest_distance', 'pri_sch_nearest_distance', 'sec_sch_nearest_dist', 'days_from_first_transaction', 'month_sin', 'month_cos', 'remaining_lease', 'property_age_at_sale', 'storey_mid_ratio', 'log_mrt_dist', 'log_mall_dist', 'log_hawker_dist', 'log_bus_dist', 'log_pri_sch_dist', 'log_sec_sch_dist', 'accessibility_score', 'geo_cluster_prob']


In [76]:
# Step 3: remaining non-continuous numeric columns.
other_numeric_cols = [
    col for col in numeric_cols
    if col not in continuous_cols and col not in engineered_numeric_cols
]

print(f'Remaining numeric columns ({len(other_numeric_cols)}):')
print(other_numeric_cols)

Remaining numeric columns (32):
['lease_commence_date', 'Tranc_Year', 'Tranc_Month', 'mid_storey', 'lower', 'upper', 'mid', 'hdb_age', 'max_floor_lvl', 'year_completed', 'total_dwelling_units', '2room_sold', '3room_sold', '4room_sold', '5room_sold', 'exec_sold', 'multigen_sold', 'studio_apartment_sold', '1room_rental', '2room_rental', '3room_rental', 'other_room_rental', 'Mall_Within_500m', 'Mall_Within_1km', 'Mall_Within_2km', 'Hawker_Within_500m', 'Hawker_Within_1km', 'Hawker_Within_2km', 'hawker_food_stalls', 'hawker_market_stalls', 'vacancy', 'cutoff_point']


## 5. Building the Preprocessing Pipeline

This is the main sklearn pipeline pattern used by the rest of the notebook. Raw feature data goes into `Pipeline`; `transaction_date_features` creates `days_from_first_transaction`, `spatial_index_features` creates S2/geohash area labels for each coordinate pair, `drop_columns` removes raw IDs and raw coordinate columns, and the `preprocessor` step scales/imputes numeric columns while passing imputed categorical columns through for CatBoost-native handling. The final `model` step receives categorical feature indices through CatBoost's `cat_features` parameter.


In [77]:
# Continuous numeric features are standardized. Raw latitude/longitude columns
# are intentionally excluded here because they are converted into spatial
# index features and dropped before preprocessing.
std_scale_cols = [
    'floor_area_sqm',
    'Mall_Nearest_Distance',
    'Hawker_Nearest_Distance',
    'mrt_nearest_distance',
    'bus_stop_nearest_distance',
    'pri_sch_nearest_distance',
    'sec_sch_nearest_dist',
    'days_from_first_transaction',
    'month_sin',
    'month_cos',
    'remaining_lease',
    'property_age_at_sale',
    'storey_mid_ratio',
    'log_mrt_dist',
    'log_mall_dist',
    'log_hawker_dist',
    'log_bus_dist',
    'log_pri_sch_dist',
    'log_sec_sch_dist',
    'accessibility_score',
    'geo_cluster_prob',
]

minmax_scale_cols = [
    '1room_sold',
    'bus_interchange',
    'mrt_interchange',
    'pri_sch_affiliation',
    'affiliation',
    'lease_commence_date',
    'Tranc_Year',
    'Tranc_Month',
    'mid_storey',
    'lower',
    'upper',
    'mid',
    'hdb_age',
    'max_floor_lvl',
    'year_completed',
    'total_dwelling_units',
    '2room_sold',
    '3room_sold',
    '4room_sold',
    '5room_sold',
    'exec_sold',
    'multigen_sold',
    'studio_apartment_sold',
    '1room_rental',
    '2room_rental',
    '3room_rental',
    'other_room_rental',
    'Mall_Within_500m',
    'Mall_Within_1km',
    'Mall_Within_2km',
    'Hawker_Within_500m',
    'Hawker_Within_1km',
    'Hawker_Within_2km',
    'hawker_food_stalls',
    'hawker_market_stalls',
    'vacancy',
    'cutoff_point',
    'transaction_quarter',
    'is_new_flat',
    'is_old_flat',
    'flat_type_ordinal',
    'mrt_walkable',
    'bus_walkable',
    'mall_walkable',
    'hawker_walkable',
    'pri_school_walkable',
    'sec_school_walkable',
    'integrated_transport_access',
    'geo_noise_flag',
]

# Categorical and geospatial bucket columns are imputed and passed through as
# strings. CatBoost then handles them natively via cat_features, including its
# own one-hot/target-statistics logic controlled by one_hot_max_size.
non_geo_categorical_cols = [col for col in categorical_cols if col not in spatial_index_cols]
catboost_categorical_cols = non_geo_categorical_cols + spatial_index_cols

std_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler()),
])

minmax_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', MinMaxScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
])

preprocessor = ColumnTransformer(transformers=[
    ('std', std_pipeline, std_scale_cols),
    ('minmax', minmax_pipeline, minmax_scale_cols),
    ('cat', categorical_pipeline, catboost_categorical_cols),
])

preprocessed_feature_names = std_scale_cols + minmax_scale_cols + catboost_categorical_cols
cat_feature_indices = list(range(
    len(std_scale_cols) + len(minmax_scale_cols),
    len(preprocessed_feature_names),
))


def make_regression_pipeline(model):
    return Pipeline(steps=[
        ('transaction_date_features', TransactionDateFeatures()),
        ('spatial_index_features', SpatialIndexFeatures()),
        ('geo_cluster_features', GeoClusterFeatures()),
        ('drop_columns', DropColumns(drop_cols)),
        ('preprocessor', preprocessor),
        ('model', model),
    ])


preprocessing_summary = pd.DataFrame({
    'group': ['transaction_date_features', 'spatial_index_features', 'geo_cluster_features', 'dropped', 'imputation', 'catboost_native_categorical', 'binary', 'standard_scaled_continuous', 'minmax_scaled_numeric'],
    'column_count': [len(TransactionDateFeatures().engineered_features), len(spatial_index_cols), len(GeoClusterFeatures().engineered_features), len(drop_cols), X.drop(columns=drop_cols, errors='ignore').isna().sum().gt(0).sum(), len(catboost_categorical_cols), len(binary_cols), len(std_scale_cols), len(minmax_scale_cols)],
})

cat_feature_indices[:5], cat_feature_indices[-5:], preprocessing_summary


([70, 71, 72, 73, 74],
 [123, 124, 125, 126, 127],
                          group  column_count
 0    transaction_date_features            28
 1       spatial_index_features            40
 2         geo_cluster_features             3
 3                      dropped            19
 4                   imputation             7
 5  catboost_native_categorical            58
 6                       binary             5
 7   standard_scaled_continuous            21
 8        minmax_scaled_numeric            49)

## 6. Train-Test Split

Create the held-out test split first. Cross-validation and hyperparameter tuning below use only `X_train` and `Y_train`, so the test split remains untouched until final evaluation.


In [78]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.1, random_state=42
)

X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

((135570, 76), (15064, 76), (135570,), (15064,))

## 7. Helper Function for Regression Metrics

This helper keeps the evaluation output consistent across CatBoost model runs.


In [79]:
def regression_metrics(model_name, y_train, y_train_pred, y_test, y_test_pred):
    return pd.DataFrame({
        'model': [model_name, model_name],
        'split': ['train', 'test'],
        'MAE': [
            mean_absolute_error(y_train, y_train_pred),
            mean_absolute_error(y_test, y_test_pred),
        ],
        'RMSE': [
            mean_squared_error(y_train, y_train_pred) ** 0.5,
            mean_squared_error(y_test, y_test_pred) ** 0.5,
        ],
        'R2': [
            r2_score(y_train, y_train_pred),
            r2_score(y_test, y_test_pred),
        ],
    })

## 8. Fast Baseline CatBoost Fit

Fit one baseline CatBoost model on the train split and evaluate it on the held-out test split. This is much faster than 10-fold cross-validation and is useful while iterating on feature engineering.


In [82]:
catboost_fast_baseline_pipeline = make_regression_pipeline(
    CatBoostRegressor(
        loss_function='RMSE',
        cat_features=cat_feature_indices,
        random_seed=42,
        verbose=0,
        allow_writing_files=False,
        thread_count=-1,
    )
)

catboost_fast_baseline_pipeline.fit(X_train, Y_train)

Y_train_pred_catboost_fast = catboost_fast_baseline_pipeline.predict(X_train)
Y_test_pred_catboost_fast = catboost_fast_baseline_pipeline.predict(X_test)

catboost_fast_baseline_metrics = regression_metrics(
    'CatBoost Fast Baseline',
    Y_train,
    Y_train_pred_catboost_fast,
    Y_test,
    Y_test_pred_catboost_fast,
)

catboost_fast_baseline_metrics


KeyboardInterrupt: 

## 9. Baseline CatBoost with 10-Fold Cross-Validation

Run 10-fold cross-validation on the training split only. This estimates baseline model stability without using the held-out test split.


In [81]:
# Use an explicit KFold loop instead of sklearn.cross_validate.
# sklearn.cross_validate clones the estimator, and CatBoost mutates/normalizes
# cat_features internally, which can trigger a clone-safety RuntimeError.
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
cv_metric_rows = []

for fold, (train_idx, valid_idx) in enumerate(kfold.split(X_train), start=1):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_valid = X_train.iloc[valid_idx]
    Y_fold_train = Y_train.iloc[train_idx]
    Y_fold_valid = Y_train.iloc[valid_idx]

    fold_pipeline = make_regression_pipeline(
        CatBoostRegressor(
            loss_function='RMSE',
            cat_features=cat_feature_indices,
            random_seed=42,
            verbose=0,
            allow_writing_files=False,
            thread_count=1,
        )
    )

    fold_pipeline.fit(X_fold_train, Y_fold_train)
    Y_fold_pred = fold_pipeline.predict(X_fold_valid)

    cv_metric_rows.append({
        'fold': fold,
        'MAE': mean_absolute_error(Y_fold_valid, Y_fold_pred),
        'RMSE': np.sqrt(mean_squared_error(Y_fold_valid, Y_fold_pred)),
        'R2': r2_score(Y_fold_valid, Y_fold_pred),
    })

catboost_cv_metrics = pd.DataFrame(cv_metric_rows)
catboost_cv_summary = catboost_cv_metrics[['MAE', 'RMSE', 'R2']].agg(['mean', 'std'])
catboost_cv_summary


KeyboardInterrupt: 

## 10. CatBoost Hyperparameter Tuning with Built-In Grid Search

CatBoost's built-in `grid_search()` tunes the model after the training split has been transformed by sklearn. Numeric features are scaled/imputed by sklearn, while regular categorical columns and engineered S2/geohash buckets plus HDBSCAN cluster labels are passed through as strings and handled by CatBoost through `cat_features`. This tuning pass follows three rules: tune learning rate with iterations and early stopping, keep tree complexity within a controlled range, and tune randomness/sampling noise for overfitting control.


In [ ]:
catboost_tuning_preprocessor = Pipeline(steps=[
    ('transaction_date_features', TransactionDateFeatures()),
    ('spatial_index_features', SpatialIndexFeatures()),
    ('geo_cluster_features', GeoClusterFeatures()),
    ('drop_columns', DropColumns(drop_cols)),
    ('preprocessor', preprocessor),
])

X_train_tuned = catboost_tuning_preprocessor.fit_transform(X_train)
X_test_tuned = catboost_tuning_preprocessor.transform(X_test)

# CatBoost receives numeric columns plus imputed categorical string columns.
# These transformed column positions tell CatBoost which columns are categorical.
cat_features = cat_feature_indices

catboost_param_grid = {
    # Learning rate + iterations: lower learning rates need more trees.
    'learning_rate': [0.03, 0.06, 0.1],
    'iterations': [500, 1500, 3000],

    # Tree complexity.
    'depth': [4, 6, 8],
    'l2_leaf_reg': [1, 3, 10],

    # Noise control.
    'random_strength': [1, 10, 20],
    'bagging_temperature': [0, 0.5, 1],

    # CatBoost-native categorical threshold.
    'one_hot_max_size': [2, 10],
}

catboost_grid_model = CatBoostRegressor(
    loss_function='RMSE',
    bootstrap_type='Bayesian',
    cat_features=cat_features,
    od_type='Iter',
    od_wait=100,
    use_best_model=False,
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
    thread_count=-1,
)

catboost_grid_search = catboost_grid_model.grid_search(
    catboost_param_grid,
    X=X_train_tuned,
    y=Y_train,
    cv=kfold,
    partition_random_seed=42,
    shuffle=True,
    refit=True,
    verbose=False,
)

best_catboost_params = catboost_grid_search['params']

best_catboost_pipeline = make_regression_pipeline(
    CatBoostRegressor(
        loss_function='RMSE',
        bootstrap_type='Bayesian',
        cat_features=cat_features,
        od_type='Iter',
        od_wait=100,
        use_best_model=True,
        random_seed=42,
        verbose=0,
        allow_writing_files=False,
        thread_count=-1,
        **best_catboost_params,
    )
)

best_catboost_pipeline.fit(
    X_train,
    Y_train,
    model__eval_set=(X_test_tuned, Y_test),
)

Y_train_pred_catboost_best = best_catboost_pipeline.predict(X_train)
Y_test_pred_catboost_best = best_catboost_pipeline.predict(X_test)

best_catboost_metrics = regression_metrics(
    'Tuned CatBoost',
    Y_train,
    Y_train_pred_catboost_best,
    Y_test,
    Y_test_pred_catboost_best,
)

best_catboost_params, best_catboost_metrics


## 11. Model Comparison

Compare the baseline and tuned CatBoost models on the held-out test split.


In [ ]:
model_comparison = pd.concat(
    [catboost_fast_baseline_metrics, best_catboost_metrics],
    ignore_index=True,
)

model_comparison.sort_values(['split', 'RMSE'])


## 12. Fit Final Model, Predict Test Data, and Save Model

Fit a final CatBoost pipeline on all labeled training data using the best grid-search parameters. Then predict `df_test` and save the fitted pipeline for reuse.


In [ ]:
from pathlib import Path
import cloudpickle

catboost_final_pipeline = make_regression_pipeline(
    CatBoostRegressor(
        loss_function='RMSE',
        bootstrap_type='Bayesian',
        cat_features=cat_feature_indices,
        od_type='Iter',
        od_wait=100,
        use_best_model=False,
        random_seed=42,
        verbose=0,
        allow_writing_files=False,
        thread_count=-1,
        **best_catboost_params,
    )
)

catboost_final_pipeline.fit(X, Y)

## Run predict on test set:

df_test_predictions = catboost_final_pipeline.predict(df_test)
catboost_test_predictions = pd.DataFrame({
    'id': df_test['id'].to_numpy(),
    'resale_price': df_test_predictions,
})

model_dir = Path('models')
model_dir.mkdir(exist_ok=True)
catboost_model_path = model_dir / 'catboost_grid_search_final.pkl'
with open(catboost_model_path, 'wb') as file:
    cloudpickle.dump(catboost_final_pipeline, file)

catboost_model_path, catboost_test_predictions.head()


## 13. Save Kaggle Submission CSV


In [ ]:
df_submission = pd.merge(
    df_sub[['Id']],
    catboost_test_predictions,
    left_on='Id',
    right_on='id',
    how='left',
)
df_submission.drop(columns=['id'], inplace=True)
df_submission.rename(columns={'resale_price': 'Predicted'}, inplace=True)
df_submission.to_csv('submission/submission_catboost_v3.csv', index=False)

df_submission.head()

In [ ]:
# Integration Point 2: CatBoost -> SHAP explanations
# CatBoost was fitted on the transformed pipeline output. This output contains
# scaled numeric columns plus categorical string columns handled natively by CatBoost.
import shap
import matplotlib.pyplot as plt

final_model = catboost_final_pipeline.named_steps['model']

shap_sample_size = min(2000, len(X_test))
X_shap_raw = X_test.sample(n=shap_sample_size, random_state=42)
X_shap_transformed = catboost_final_pipeline[:-1].transform(X_shap_raw)
X_shap_df = pd.DataFrame(
    X_shap_transformed,
    columns=preprocessed_feature_names,
    index=X_shap_raw.index,
)

X_shap_pool = Pool(
    X_shap_df,
    cat_features=cat_feature_indices,
)

explainer = shap.TreeExplainer(final_model)

print('Calculating SHAP values...')
shap_values = explainer.shap_values(X_shap_pool)
print(f'SHAP values calculated for {shap_values.shape[0]} predictions')
print(f'Each prediction explained by {shap_values.shape[1]} transformed features')

catboost_importance = final_model.get_feature_importance()
shap_importance_values = np.mean(np.abs(shap_values), axis=0)

shap_importance = (
    pd.DataFrame({
        'Feature': preprocessed_feature_names,
        'SHAP_MeanAbs': shap_importance_values,
        'CatBoost_Importance': catboost_importance,
    })
    .sort_values('SHAP_MeanAbs', ascending=False)
    .reset_index(drop=True)
)

plt.figure(figsize=(12, 8))
shap.summary_plot(
    shap_values,
    X_shap_df,
    feature_names=preprocessed_feature_names,
    plot_type='bar',
    max_display=10,
    show=False,
)
plt.title('Global Feature Importance (SHAP)', fontweight='bold', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

print('CatBoost Feature Importance Rankings (Top 10):')
print('=' * 70)
print(f"{'Rank':<6}{'Feature':<50}{'Importance':>12}")
print('-' * 70)
catboost_ranking = shap_importance.sort_values('CatBoost_Importance', ascending=False).head(10).reset_index(drop=True)
for rank, row in catboost_ranking.iterrows():
    feature = row['Feature']
    if len(feature) > 47:
        feature = feature[:44] + '...'
    print(f"{rank + 1:<6}{feature:<50}{row['CatBoost_Importance']:>12.1f}")

shap_importance.head(30)
